# CTDenoiser — Colab Notebook

End-to-end workflow in one place:

| Part | What happens |
|------|-------------|
| **0 — Setup** | Mount Drive, clone/update repo, install deps |
| **1 — Data prep** | Download TCIA LDCT data, build `ldct_cache.h5` |
| **2 — Smoke test** | Verify everything works on synthetic data |
| **3 — Train** | Train RED-CNN and CTformer on real data |
| **4 — W&B sweep** | Bayesian HP search over both architectures |

**Skip Part 1** if you already have `ldct_cache.h5` on Drive.

---
## Part 0 — Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL  = 'https://github.com/tsereda/CTDenoiser.git'
BRANCH    = 'main'
REPO_ROOT = '/content/drive/MyDrive/CTDenoiser'

if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
    os.makedirs(REPO_ROOT, exist_ok=True)
    !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
    print('Cloned.')
else:
    print('Repo already exists, pulling latest.')

%cd {REPO_ROOT}
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}

In [ ]:
%cd {REPO_ROOT}
!pip install -q -r requirements.txt

In [ ]:
%cd {REPO_ROOT}
!pytest -q

---
## Part 1 — TCIA Data Download & HDF5 Cache

Downloads **LDCT-and-Projection-data** from TCIA (public, no account required),
converts DICOM → Hounsfield-unit arrays, and writes `ldct_cache.h5` to Google Drive.

**Expected runtime:** 30–90 min &nbsp;|&nbsp; **Disk:** ~4 GB per patient pair

```
ldct_cache.h5
  L004_low    (num_slices, 512, 512)  float32  raw HU
  L004_full   (num_slices, 512, 512)  float32  raw HU
  ...
```

> **Skip this part** if `ldct_cache.h5` already exists on your Drive.

In [ ]:
!pip install -q tcia_utils pydicom

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
DRIVE_CACHE   = '/content/drive/MyDrive/ldct_cache.h5'   # final destination
LOCAL_CACHE   = '/content/ldct_cache.h5'                 # fast scratch disk
DICOM_DIR     = '/content/ldct_dicom'                    # scratch for DICOMs

MAX_PATIENTS  = 10          # each ~4 GB; set None to grab all ~50
LOW_DOSE_TAG  = 'Low Dose Images'
FULL_DOSE_TAG = 'Full Dose Images'
BODY_PART     = 'ABDOMEN'
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
from tcia_utils import nbia
import pandas as pd

print('Querying TCIA for LDCT-and-Projection-data series...')
series = nbia.getSeries(collection='LDCT-and-Projection-data')
df = pd.DataFrame(series)

print(f'Total series: {len(df)}')
print('Body parts:', df['BodyPartExamined'].unique())
print('Descriptions:', df['SeriesDescription'].unique()[:10])

In [ ]:
imgs = df[
    (df['BodyPartExamined'] == BODY_PART) &
    (df['SeriesDescription'].isin([LOW_DOSE_TAG, FULL_DOSE_TAG]))
].copy()

by_patient = imgs.groupby('PatientID')['SeriesDescription'].apply(set)
paired_ids = by_patient[
    by_patient.apply(lambda x: LOW_DOSE_TAG in x and FULL_DOSE_TAG in x)
].index.tolist()

print(f'Patients with paired low+full dose: {len(paired_ids)}')
print('Sample IDs:', paired_ids[:5])

if MAX_PATIENTS:
    paired_ids = paired_ids[:MAX_PATIENTS]
    print(f'Limiting to {MAX_PATIENTS} patients.')

paired_series = imgs[imgs['PatientID'].isin(paired_ids)]
print(f'Series to download: {len(paired_series)}')
paired_series[['PatientID', 'SeriesDescription', 'SeriesInstanceUID']].head(10)

In [ ]:
import os

os.makedirs(DICOM_DIR, exist_ok=True)
print(f'Downloading {len(paired_series)} series to {DICOM_DIR}...')

nbia.downloadSeries(
    series_data=paired_series['SeriesInstanceUID'].tolist(),
    path=DICOM_DIR,
    input_type='list'
)
print('Download complete.')

In [ ]:
import glob

dcm_files = glob.glob(os.path.join(DICOM_DIR, '**', '*.dcm'), recursive=True)
print(f'Total DICOM files found: {len(dcm_files)}')

series_dirs = sorted(set(os.path.dirname(f) for f in dcm_files))
print(f'Series folders: {len(series_dirs)}')
for d in series_dirs[:6]:
    n = len(glob.glob(os.path.join(d, '*.dcm')))
    print(f'  {os.path.basename(d)[:60]}  ({n} slices)')

In [ ]:
import pydicom
import numpy as np
import h5py
from pathlib import Path

def read_series_hu(series_dir: str) -> np.ndarray:
    """Load all DICOM slices in a directory, sort by position, return HU array."""
    dcms = sorted(
        Path(series_dir).glob('*.dcm'),
        key=lambda p: float(
            pydicom.dcmread(str(p), stop_before_pixels=True).ImagePositionPatient[2]
        )
    )
    if not dcms:
        raise ValueError(f'No .dcm files in {series_dir}')
    slices = []
    for p in dcms:
        ds = pydicom.dcmread(str(p))
        hu = ds.pixel_array.astype(np.float32) * float(ds.RescaleSlope) + float(ds.RescaleIntercept)
        slices.append(hu)
    return np.stack(slices, axis=0)  # (num_slices, H, W)


uid_meta = {
    row['SeriesInstanceUID']: (row['PatientID'], row['SeriesDescription'])
    for _, row in paired_series.iterrows()
}

series_dirs = [d for d in Path(DICOM_DIR).iterdir() if d.is_dir()]

print(f'Building HDF5 cache at {LOCAL_CACHE} ...')
failed = []
with h5py.File(LOCAL_CACHE, 'w') as hf:
    for sdir in sorted(series_dirs):
        uid = sdir.name
        if uid not in uid_meta:
            continue
        pid, desc = uid_meta[uid]
        dose_key = 'low' if desc == LOW_DOSE_TAG else 'full'
        h5_key   = f'{pid}_{dose_key}'
        try:
            arr = read_series_hu(str(sdir))
            hf.create_dataset(h5_key, data=arr, compression='lzf')
            print(f'  wrote {h5_key:30s}  {arr.shape}  [{arr.min():.0f}, {arr.max():.0f}] HU')
        except Exception as e:
            print(f'  SKIP {h5_key}: {e}')
            failed.append(h5_key)

print(f'\nDone. Failed: {failed if failed else "none"}')

In [ ]:
with h5py.File(LOCAL_CACHE, 'r') as hf:
    keys = sorted(hf.keys())
    patients = sorted({k.split('_')[0] for k in keys})
    print(f'{len(patients)} patients, {len(keys)} datasets')
    print('Patients:', patients)
    print()
    for k in keys[:6]:
        print(f'  {k:30s}  {hf[k].shape}')

In [ ]:
import shutil

print(f'Copying {LOCAL_CACHE} -> {DRIVE_CACHE} ...')
shutil.copy(LOCAL_CACHE, DRIVE_CACHE)
print('Saved to Drive.')

---
## Part 2 — Smoke Test (Synthetic Data)

Verifies the training loop runs end-to-end without needing real CT data.

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model redcnn --epochs 1 --synthetic-len 32
print('RED-CNN smoke test passed.')

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model ctformer --epochs 1 --synthetic-len 32
print('CTformer smoke test passed.')

---
## Part 3 — Train RED-CNN and CTformer on Real Data

Copies the HDF5 cache to fast local disk, then trains both models sequentially.
Adjust `--epochs` and `--batch-size` to your GPU budget.

In [ ]:
import os, shutil

DRIVE_CACHE = '/content/drive/MyDrive/ldct_cache.h5'
LOCAL_CACHE = '/content/ldct_cache.h5'

if os.path.exists(DRIVE_CACHE) and not os.path.exists(LOCAL_CACHE):
    print('Copying cache to local disk for faster I/O...')
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    print('Done.')

assert os.path.exists(LOCAL_CACHE), (
    'ldct_cache.h5 not found — run Part 1 first, or place the file at ' + DRIVE_CACHE
)

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model redcnn \
    --h5-cache /content/ldct_cache.h5 \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model ctformer \
    --h5-cache /content/ldct_cache.h5 \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

---
## Part 4 — Weights & Biases Hyperparameter Sweep

Bayesian search over learning rate, batch size, patch size, **and model architecture**
(RED-CNN vs CTformer). Each run is logged to your W&B project so you can compare
them side-by-side.

**Before running:** log in to W&B once with `wandb login` or set `WANDB_API_KEY`.

In [ ]:
import wandb
wandb.login()   # prompts for API key if not already set

In [ ]:
SWEEP_PROJECT = 'ctdenoiser'

# How many total runs to launch (increase for a thorough search)
SWEEP_COUNT = 20

# Path to the HDF5 cache (set to None to use synthetic data)
SWEEP_H5_CACHE = '/content/ldct_cache.h5'

sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val/psnr', 'goal': 'maximize'},
    'parameters': {
        'model':      {'values': ['redcnn', 'ctformer']},
        'lr':         {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-3},
        'batch_size': {'values': [8, 16, 32]},
        'patch_size': {'values': [64, 96]},
        'epochs':     {'value': 10},
    },
}

sweep_id = wandb.sweep(sweep_config, project=SWEEP_PROJECT)
print(f'Sweep created: {sweep_id}')

In [ ]:
import sys, os
sys.path.insert(0, REPO_ROOT)

import torch
from torch.utils.data import DataLoader
import wandb

from ctdenoiser.models import CTformer, REDCNN
from ctdenoiser.data.dataset import HDF5CTDataset, SyntheticCTDataset
from ctdenoiser.inference import overlapped_inference
from ctdenoiser.metrics import psnr, rmse, ssim

MODELS = {'ctformer': CTformer, 'redcnn': REDCNN}


def train_run():
    """Single W&B sweep run: build loaders, train, log metrics."""
    run = wandb.init()
    cfg = run.config

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = MODELS[cfg.model]().to(device)

    wandb.config.update({'device': str(device)}, allow_val_change=True)

    # ── Data ────────────────────────────────────────────────────────────────
    h5_path = SWEEP_H5_CACHE
    if h5_path and os.path.exists(h5_path):
        train_pids, val_pids = HDF5CTDataset.split_patients(h5_path, val_fraction=0.2, seed=0)
        train_ds = HDF5CTDataset(h5_path, train_pids, patch_size=cfg.patch_size, train=True)
        val_ds   = HDF5CTDataset(h5_path, val_pids,   patch_size=cfg.patch_size, train=False)
        full_slice_eval = True
    else:
        print('H5 cache not found — using synthetic data.')
        train_ds = SyntheticCTDataset(length=128, patch_size=cfg.patch_size)
        val_ds   = SyntheticCTDataset(length=32,  patch_size=cfg.patch_size)
        full_slice_eval = False

    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=1, shuffle=False
    )

    # ── Training ────────────────────────────────────────────────────────────
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    criterion = torch.nn.MSELoss()
    n_train   = len(train_loader.dataset)

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        running = 0.0
        for low, full in train_loader:
            low, full = low.to(device), full.to(device)
            optimizer.zero_grad()
            loss = criterion(model(low), full)
            loss.backward()
            optimizer.step()
            running += loss.item() * low.size(0)

        train_loss = running / n_train
        wandb.log({'epoch': epoch, 'train/loss': train_loss})

    # ── Validation ──────────────────────────────────────────────────────────
    model.eval()
    n, p, s, r = 0, 0.0, 0.0, 0.0
    with torch.no_grad():
        for low, full in val_loader:
            low, full = low.to(device), full.to(device)
            if full_slice_eval:
                pred = overlapped_inference(
                    model, low,
                    patch_size=cfg.patch_size,
                    margin=cfg.patch_size // 4,
                ).clamp(0.0, 1.0)
            else:
                pred = model(low).clamp(0.0, 1.0)
            bs = low.size(0)
            p += psnr(pred, full) * bs
            s += ssim(pred, full) * bs
            r += rmse(pred, full) * bs
            n += bs

    wandb.log({
        'val/psnr': p / n,
        'val/ssim': s / n,
        'val/rmse': r / n,
    })
    run.finish()


# Launch the sweep agent
wandb.agent(sweep_id, function=train_run, count=SWEEP_COUNT)

### Viewing Results

Open [wandb.ai](https://wandb.ai) → your project `ctdenoiser` → **Sweeps** tab.

Useful views:
- **Parallel coordinates** — see which HP combos dominate on val/psnr
- **Scatter plot** `model` vs `val/psnr` — direct RED-CNN vs CTformer comparison
- **Importance** panel — which hyperparameter matters most